In [5]:
import json
import pandas as pd

file_name = "acndata_sessions.json"

print("Lecture et réparation du fichier...")
with open(file_name, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

# On cherche la dernière accolade fermante d'une session complète
last_brace = raw_text.rfind("}")

if last_brace != -1:
    # On isole jusqu'à la dernière session complète
    truncated = raw_text[:last_brace + 1].strip()
    
    # Si la chaîne se termine par une virgule parasite, on l'enlève
    if truncated.endswith(","):
        truncated = truncated[:-1]
        
    # On referme le tableau _items et l'objet JSON
    repaired_json = truncated + "\n  ]\n}"
    
    try:
        data = json.loads(repaired_json)
        sessions = data.get("_items", [])
        print(f"✅ Succès ! {len(sessions)} sessions récupérées.")
    except json.JSONDecodeError:
        # Repli si la racine était coupée différemment : on extrait directement le bloc _items
        items_start = raw_text.find('"_items": [')
        content = raw_text[items_start + len('"_items": ['):last_brace + 1]
        valid_items_str = "[" + content.rstrip(",") + "]"
        sessions = json.loads(valid_items_str)
        print(f"✅ Succès via extraction directe ! {len(sessions)} sessions récupérées.")

df = pd.DataFrame(sessions)
display(df.head(3))

Lecture et réparation du fichier...
✅ Succès ! 14599 sessions récupérées.


,_id,clusterID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs
0,5bc93906f9af8b0dc677d717,0039,"Sat, 01 Sep 2018 00:11:13 GMT","Sat, 01 Sep 2018 03:06:31 GMT","Sat, 01 Sep 2018 02:11:40 GMT",6.120,2_39_138_29_2018-09-01 00:11:13.309447,0002,CA-304,2-39-138-29,America/Los_Angeles,NaN,None
1,5bc93906f9af8b0dc677d718,0039,"Sat, 01 Sep 2018 00:17:18 GMT","Sat, 01 Sep 2018 02:42:03 GMT","Sat, 01 Sep 2018 02:19:38 GMT",13.235,2_39_79_379_2018-09-01 00:17:17.628436,0002,CA-327,2-39-79-379,America/Los_Angeles,NaN,None
2,5bc93906f9af8b0dc677d719,0039,"Sat, 01 Sep 2018 00:23:49 GMT","Sat, 01 Sep 2018 01:41:04 GMT","Sat, 01 Sep 2018 01:23:19 GMT",5.890,2_39_125_21_2018-09-01 00:23:49.232101,0002,CA-311,2-39-125-21,America/Los_Angeles,NaN,None


In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# 1. Traitement des timestamps
df["connectionTime"] = pd.to_datetime(df["connectionTime"], utc=True)
df["disconnectTime"] = pd.to_datetime(df["disconnectTime"], utc=True)
df["doneChargingTime"] = pd.to_datetime(df["doneChargingTime"], utc=True)

# 2. Durées (en heures)
df["plugged_hrs"] = (df["disconnectTime"] - df["connectionTime"]).dt.total_seconds() / 3600.0
# Durée pendant laquelle la borne a réellement délivré de l'énergie
df["charging_hrs"] = (df["doneChargingTime"] - df["connectionTime"]).dt.total_seconds() / 3600.0
# En cas d'enregistrement manquant ou incohérent, on borne à 0
df["charging_hrs"] = df["charging_hrs"].clip(lower=0)

# 3. Puissance moyenne effective délivrée (kW)
# Évite la division par zéro avec un epsilon
df["avg_kw_delivered"] = df["kWhDelivered"] / df["charging_hrs"].replace(0, np.nan)

# 4. Signatures d'anomalies de session :
# A. Charge ultra-courte avortée (< 5 minutes de charge réelle)
df["short_cutoff"] = (df["charging_hrs"] < (5 / 60)) & (df["plugged_hrs"] > (15 / 60))

# B. Faible énergie (< 1.5 kWh) malgré un branchement prolongé (> 30 min)
df["low_energy_cutoff"] = (df["kWhDelivered"] < 1.5) & (df["plugged_hrs"] > 0.5)

# C. Puissance anormale (débit < 1.5 kW sur une borne censée délivrer 3.3 à 6.6 kW)
df["underpower_charge"] = (df["avg_kw_delivered"] < 1.5) & (df["charging_hrs"] > 0.25)

# 5. Agrégation par borne
bornes = df.groupby("stationID").agg(
    total_sessions=("sessionID", "count"),
    total_kwh=("kWhDelivered", "sum"),
    avg_kwh=("kWhDelivered", "mean"),
    short_cutoff_count=("short_cutoff", "sum"),
    low_energy_count=("low_energy_cutoff", "sum"),
    underpower_count=("underpower_charge", "sum"),
    avg_charging_hrs=("charging_hrs", "mean")
).reset_index()

# Calcul des ratios
bornes["short_cutoff_rate"] = bornes["short_cutoff_count"] / bornes["total_sessions"]
bornes["low_energy_rate"] = bornes["low_energy_count"] / bornes["total_sessions"]
bornes["underpower_rate"] = bornes["underpower_count"] / bornes["total_sessions"]

# Filtrer sur les bornes avec une activité significative (> 25 sessions)
bornes_actives = bornes[bornes["total_sessions"] >= 25].copy()

# 6. Score d'anomalie IA centré sur les défaillances
features = [
    "short_cutoff_rate", 
    "low_energy_rate", 
    "underpower_rate", 
    "avg_kwh", 
    "avg_charging_hrs"
]
X = bornes_actives[features].fillna(0)

# On normalise les features pour éviter que avg_kwh n'écrase les pourcentages
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(contamination=0.15, random_state=42)
bornes_actives["anomaly_flag"] = iso.fit_predict(X_scaled)
bornes_actives["risk_score"] = -iso.decision_function(X_scaled)

# Classement
top_priorites = bornes_actives.sort_values(by="risk_score", ascending=False).reset_index(drop=True)

colonnes_vue = [
    "stationID", 
    "total_sessions", 
    "short_cutoff_rate", 
    "low_energy_rate", 
    "underpower_rate", 
    "avg_kwh", 
    "risk_score"
]
top_priorites[colonnes_vue].head(10)

,stationID,total_sessions,short_cutoff_rate,low_energy_rate,underpower_rate,avg_kwh,risk_score
0,2-39-78-362,224,0.000000,0.156250,0.584821,8.521242,0.179125
1,2-39-78-367,244,0.028689,0.176230,0.233607,8.080084,0.149074
2,2-39-78-360,292,0.017123,0.239726,0.205479,8.518762,0.133152
3,2-39-78-361,276,0.003623,0.271739,0.289855,7.103453,0.127308
4,2-39-92-442,263,0.000000,0.079848,0.057034,15.930661,0.064693
5,2-39-78-366,388,0.002577,0.131443,0.273196,9.948854,0.040399
6,2-39-78-365,342,0.008772,0.093567,0.236842,8.434055,0.037389
7,2-39-127-561,36,0.000000,0.000000,0.166667,11.241722,0.025195
8,2-39-78-364,231,0.004329,0.138528,0.186147,8.838204,-0.013566
9,2-39-129-17,532,0.001880,0.090226,0.052632,8.779317,-0.024964
